# Oxford Flowers GAN — SageMaker Training Launcher

Run this notebook from SageMaker Studio or any environment with `boto3` and `sagemaker` SDK.

**What it does:**
1. Configures a PyTorch training job pointing at `train.py`
2. Submits to a GPU instance (default: `ml.g4dn.xlarge` — ~\$0.74/hr, ~35 min run ≈ \$0.43)
3. Optionally uses Spot instances for ~70% cost reduction
4. Downloads model artifacts and generated images from S3

In [ ]:
!pip install -q sagemaker boto3 --upgrade

## 1. Session Setup

In [ ]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch

session    = sagemaker.Session()
role       = sagemaker.get_execution_role()   # IAM role with SageMaker + S3 permissions
bucket     = session.default_bucket()         # auto-created S3 bucket
region     = session.boto_region_name
prefix     = 'oxford-flowers-gan'

print(f'Region:  {region}')
print(f'Bucket:  s3://{bucket}/{prefix}')
print(f'Role:    {role}')

## 2. (Optional) Stage Data to S3

Skip this if you want `train.py` to download the Oxford Flowers dataset directly inside the container — it will fall back to downloading from the Oxford server automatically.

Staging to S3 is faster for repeated runs since the data is already in AWS.

In [ ]:
# OPTIONAL — run this cell once to upload the dataset to S3
import torchvision
from torchvision import transforms

# Download locally first
torchvision.datasets.Flowers102('flowers-data', split='train', download=True)

# Upload to S3
data_s3_uri = session.upload_data(
    path='flowers-data/flowers-102',
    bucket=bucket,
    key_prefix=f'{prefix}/data'
)
print(f'Data uploaded to: {data_s3_uri}')

## 3. Configure Training Job

In [ ]:
# --- Instance options ---
# ml.g4dn.xlarge  — 1x T4 GPU,  4 vCPU, 16 GB RAM  ~$0.74/hr  (recommended)
# ml.g4dn.2xlarge — 1x T4 GPU,  8 vCPU, 32 GB RAM  ~$0.94/hr
# ml.p3.2xlarge   — 1x V100 GPU, 8 vCPU, 61 GB RAM  ~$3.06/hr  (fastest)

INSTANCE_TYPE  = 'ml.g4dn.xlarge'
USE_SPOT       = True   # ~70% cheaper; adds max_wait for spot availability
MAX_RUN_SECS   = 7200   # 2 hours hard limit
MAX_WAIT_SECS  = 10800  # 3 hours spot wait limit (must be >= MAX_RUN_SECS)

hyperparameters = {
    'epochs':       250,
    'lr':           0.0001,
    'beta1':        0.5,
    'batch-size':   102,
    'nz':           100,
    'ngf':          64,
    'ndf':          64,
    'image-size':   64,
    'log-interval': 25,
}

# SageMaker parses these regex patterns from stdout → CloudWatch Metrics
metric_definitions = [
    {'Name': 'D_loss', 'Regex': r'D_loss=(\S+);'},
    {'Name': 'G_loss', 'Regex': r'G_loss=(\S+);'},
    {'Name': 'D_x',    'Regex': r'D_x=(\S+);'},
    {'Name': 'D_Gz',   'Regex': r'D_Gz=(\S+);'},
]

estimator = PyTorch(
    entry_point        = 'train.py',
    source_dir         = '.',           # uploads train.py + requirements.txt
    role               = role,
    framework_version  = '2.4',
    py_version         = 'py311',
    instance_type      = INSTANCE_TYPE,
    instance_count     = 1,
    hyperparameters    = hyperparameters,
    metric_definitions = metric_definitions,
    output_path        = f's3://{bucket}/{prefix}/output',
    checkpoint_s3_uri  = f's3://{bucket}/{prefix}/checkpoints',
    use_spot_instances = USE_SPOT,
    max_run            = MAX_RUN_SECS,
    max_wait           = MAX_WAIT_SECS if USE_SPOT else None,
    base_job_name      = 'flowers-gan',
)

print(f'Estimator configured — instance: {INSTANCE_TYPE}, spot: {USE_SPOT}')
cost_est = (MAX_RUN_SECS / 3600) * (0.74 if 'g4dn.xlarge' in INSTANCE_TYPE else 3.06)
cost_est_spot = cost_est * 0.30
print(f'Max cost estimate: ${cost_est:.2f} on-demand / ${cost_est_spot:.2f} spot')

## 4. Launch Training Job

This submits the job to SageMaker. `wait=True` streams logs here; set `wait=False` to fire-and-forget and check SageMaker console.

In [ ]:
# If you staged data to S3 in step 2, pass it as a channel:
#   estimator.fit({'training': data_s3_uri}, wait=True)
# Otherwise let train.py download the dataset inside the container:

estimator.fit(wait=True)

## 5. Download Artifacts

In [ ]:
import os, tarfile

# The model.tar.gz is at estimator.model_data
model_s3_path = estimator.model_data
print(f'Model artifacts: {model_s3_path}')

os.makedirs('sm_artifacts', exist_ok=True)
local_tar = 'sm_artifacts/model.tar.gz'

boto3.client('s3').download_file(
    bucket,
    model_s3_path.replace(f's3://{bucket}/', ''),
    local_tar
)

with tarfile.open(local_tar, 'r:gz') as tar:
    tar.extractall('sm_artifacts')

print('Extracted:', os.listdir('sm_artifacts'))

In [ ]:
# --- Quick sanity check: load generator and sample images ---
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Rebuild architecture (same params as training)
import sys
sys.path.insert(0, '.')
from train import Generator

nz, ngf = 100, 64
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

netG = Generator(nc=3, nz=nz, ngf=ngf).to(device)
netG.load_state_dict(torch.load('sm_artifacts/generator.pth', map_location=device))
netG.eval()

with torch.no_grad():
    noise = torch.randn(64, nz, 1, 1, device=device)
    fake  = netG(noise).cpu()

plt.figure(figsize=(10, 10))
plt.axis('off')
plt.title('Generated Flowers — SageMaker-trained model', fontsize=13, fontweight='bold')
plt.imshow(np.transpose(vutils.make_grid(fake, padding=2, normalize=True), (1, 2, 0)))
plt.savefig('sm_artifacts/generated_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. CloudWatch Metrics

SageMaker automatically sent `D_loss`, `G_loss`, `D_x`, `D_Gz` to CloudWatch during training.

To view them:
- **SageMaker console** → Training Jobs → select job → Training metrics tab
- Or run the cell below to pull them via SDK

In [ ]:
# Pull CloudWatch metrics for the completed job
from sagemaker import TrainingJobAnalytics

job_name = estimator.latest_training_job.name
print(f'Job name: {job_name}')

for metric in ['G_loss', 'D_loss', 'D_x', 'D_Gz']:
    try:
        df = TrainingJobAnalytics(job_name, metric_names=[metric]).dataframe()
        if not df.empty:
            plt.figure(figsize=(10, 3))
            plt.plot(df['timestamp'], df['value'])
            plt.title(metric)
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f'Could not load {metric}: {e}')